data ingestion


In [1]:

from langchain_core.documents import Document

In [2]:
doc=Document(
    page_content="this is a info page to mark the start of project ",
    metadata={
        "date":"26 mar 2026"

    }
)
doc

Document(metadata={'date': '26 mar 2026'}, page_content='this is a info page to mark the start of project ')

In [3]:
from langchain_community.document_loaders import TextLoader

c:\Users\rsuma\OneDrive\Desktop\RAG\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
loader=TextLoader('knowledge_base.txt',encoding='utf-8')

In [5]:
loader

In [6]:
document=loader.load()
print(document)

[Document(metadata={'source': 'knowledge_base.txt'}, page_content="COURSE: MACHINE LEARNING (22CD62)Semester: VCredits: 04Total Pedagogy Hours: 50Assessment: CIE (50 Marks), SEE (50 Marks), Total (100 Marks)Pre-requisites: Data Structures & Algorithms, Theory of Probability, and Statistical AnalysisCourse ObjectivesDefine machine learning and relevant problemsInterpret various learning algorithms and learn from dataDifferentiate between supervised, unsupervised, and reinforcement learningApply performance evaluation parameters and model selection for ML problemsSyllabus ModulesModule 1: Fundamentals and Concept Learning (10 Hours)Introduction: Well-posed learning problems, Designing a Learning System, Perspectives and Issues in Machine LearningConcept Learning: Concept learning as search, Find-S algorithm, Candidate Elimination algorithm, and the Inductive bias of Candidate EliminationModule 2: Classification (10 Hours)Decision Tree Learning: Introduction, Representation, and appropria

In [7]:
# 1. Import required libraries
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 2. Load your text file
loader = TextLoader("knowledge_base.txt")  # change path if needed
documents = loader.load()

# 3. Split text into chunks (VERY IMPORTANT)
text_splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

# 4. Create embeddings model
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# 5. Store embeddings in vector DB (Chroma)
vector_db = Chroma.from_documents(
    docs,
    embedding_model,
    persist_directory="./chroma_db"
)

# 6. Save database (important)
vector_db.persist()

print("✅ Embedding complete and stored!")

Created a chunk of size 2648, which is longer than the specified 500
Created a chunk of size 2934, which is longer than the specified 500
C:\Users\rsuma\AppData\Local\Temp\ipykernel_11352\2150929488.py:20: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1874.34it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding complete and stored!


C:\Users\rsuma\AppData\Local\Temp\ipykernel_11352\2150929488.py:32: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_db.persist()


In [8]:
query = "what is ML syllabus"
results = vector_db.similarity_search(query, k=2)

for res in results:
    print(res.page_content)

COURSE: MACHINE LEARNING (22CD62)Semester: VCredits: 04Total Pedagogy Hours: 50Assessment: CIE (50 Marks), SEE (50 Marks), Total (100 Marks)Pre-requisites: Data Structures & Algorithms, Theory of Probability, and Statistical AnalysisCourse ObjectivesDefine machine learning and relevant problemsInterpret various learning algorithms and learn from dataDifferentiate between supervised, unsupervised, and reinforcement learningApply performance evaluation parameters and model selection for ML problemsSyllabus ModulesModule 1: Fundamentals and Concept Learning (10 Hours)Introduction: Well-posed learning problems, Designing a Learning System, Perspectives and Issues in Machine LearningConcept Learning: Concept learning as search, Find-S algorithm, Candidate Elimination algorithm, and the Inductive bias of Candidate EliminationModule 2: Classification (10 Hours)Decision Tree Learning: Introduction, Representation, and appropriate problemsAlgorithms: ID3 algorithm, Hypothesis Search Space, and 

REtrieval pipelinee

In [7]:
# =========================
# 🔐 LOAD ENV
# =========================
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("❌ GEMINI_API_KEY not found!")

# Map to what LangChain expects
os.environ["GOOGLE_API_KEY"] = api_key


# =========================
# 🔹 IMPORTS
# =========================
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda


# =========================
# 🔹 LOAD DOCUMENT
# =========================
loader = TextLoader("knowledge_base.txt")
documents = loader.load()


# =========================
# 🔹 SPLIT TEXT
# =========================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
docs = text_splitter.split_documents(documents)


# =========================
# 🔹 EMBEDDINGS (STABLE ✅)
# =========================
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)


# =========================
# 🔹 VECTOR DATABASE
# =========================
persist_dir = "./chroma_db"

vector_db = Chroma.from_documents(
    docs,
    embedding_model,
    persist_directory=persist_dir
)

vector_db.persist()


# =========================
# 🔹 RETRIEVER
# =========================
retriever = vector_db.as_retriever(search_kwargs={"k": 3})


# =========================
# 🔹 GEMINI LLM (WORKING ✅)
# =========================
llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",   # latest working model
    temperature=0.3
)


# =========================
# 🔹 HELPER FUNCTIONS
# =========================
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


def build_prompt(data):
    return f"""
You are an AI assistant. Answer ONLY using the context below.

Context:
{data["context"]}

Question:
{data["question"]}

Answer clearly and concisely:
"""


# =========================
# 🔹 RAG PIPELINE
# =========================
rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | RunnableLambda(build_prompt)
    | llm
    | StrOutputParser()
)


# =========================
# 🔹 RUN QUERY
# =========================
query = "What is this document about?"

response = rag_chain.invoke(query)

print("\n🧠 Answer:\n", response)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4548.51it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.



🧠 Answer:
 This document outlines the curriculum for Module 2 and Module 3, covering Literature Review & Citations (including bibliographic databases and technical reading) and Intellectual Property (Patents).
